# Precompute COVID Disease Projection — Wilk et al.

**Purpose:** Load Wilk 2020 COVID PBMCs via CELLxGENE Census, embed with Geneformer (Helical SDK), project into the PBMC3k healthy reference UMAP, score cosine distance to healthy centroids, export parquets to S3 + Neon provenance.

**Expected runtime:** 25–45 minutes on Colab T4 GPU (Census download + Geneformer on ~10k cells).

**Prerequisites on S3:** `v1/pbmc3k/geneformer_embeddings.parquet` and `v1/pbmc3k/pbmc_umap_reducer.joblib` from the PBMC3k healthy-reference precompute.


In [ ]:
# Cell 1 — install deps (numpy 1.x strategy, matches helical 0.0.1a* + the published PBMC3k embeddings)
# MUST run on a FRESH runtime: Runtime → Disconnect and delete runtime → reconnect.
# After this cell finishes, Runtime → Restart session, then run Cell 1.5 (verify).
!pip install --quiet \
  "numpy==1.26.4" \
  "scipy==1.13.1" \
  "pandas==2.2.2" \
  "torch==2.6.0" \
  "helical>=0.0.1a14,<1.0" \
  "cellxgene-census>=1.14,<1.17" \
  "scanpy>=1.10,<1.11" \
  "pyarrow>=15,<17" \
  "boto3>=1.34,<2.0" \
  "sqlmodel>=0.0.22,<0.1" \
  "asyncpg>=0.29,<0.30" \
  "httpx>=0.27,<0.29" \
  "nest-asyncio>=1.6,<2.0" \
  "umap-learn>=0.5,<0.6" \
  "joblib>=1.3,<2.0"
# Remove Colab's preinstalled cupy (built against numpy 2.x) — otherwise scanpy
# transitively imports cupy at runtime and Cell 1.5 throws ImportError.
!pip uninstall -y cupy cupy-cuda11x cupy-cuda12x 2>/dev/null || true
print("install done — now Runtime → Restart session (NOT delete), then run Cell 1.5")

In [ ]:
# Cell 1.5 — verify numpy 1.x strategy landed cleanly (run AFTER Runtime → Restart session)
import numpy as np, pandas as pd, scipy, torch
import cellxgene_census as cgc
from helical.models.geneformer import Geneformer, GeneformerConfig
print(f"numpy {np.__version__} | scipy {scipy.__version__} | pandas {pd.__version__} | torch {torch.__version__} | cuda {torch.cuda.is_available()}")
assert np.__version__.startswith("1.26"), f"Expected numpy 1.26.x, got {np.__version__}"
assert scipy.__version__.startswith("1.13"), f"Expected scipy 1.13.x, got {scipy.__version__}"
print("✅ ABI clean — proceed to Cell 2 (config)")

In [ ]:
# Cell 2 — config
import os
import subprocess
from pathlib import Path

DATABASE_URL = os.environ.get("DATABASE_URL", "")
S3_BUCKET = os.environ.get("S3_BUCKET", "")
S3_REGION = os.environ.get("S3_REGION", "us-east-1")
AWS_ACCESS_KEY_ID = os.environ.get("AWS_ACCESS_KEY_ID", "")
AWS_SECRET_ACCESS_KEY = os.environ.get("AWS_SECRET_ACCESS_KEY", "")
BACKEND_URL = os.environ.get("BACKEND_URL", "https://api.helical.manumustudio.com")

VERSION = "v1"
SEED = 42
N_SUBSAMPLE = 10_000
CENSUS_VERSION = "2025-11-08"
EMBED_DIM = 512

GIT_SHA = (
    subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=Path.cwd(), text=True).strip()
    if Path(".git").exists()
    else "colab-unknown"
)
print("GIT_SHA", GIT_SHA)


## CELLxGENE Census — resolve Wilk 2020 dataset

Opens the Census, finds the Wilk *Nature Medicine* 2020 PBMC atlas, and pins `dataset_id` / `collection_id` for reproducible filtering.


In [ ]:
# Cell 3 — Census exploration
import cellxgene_census as cgc

with cgc.open_soma(census_version=CENSUS_VERSION) as census:
    datasets = census["census_info"]["datasets"].read().concat().to_pandas()

mask = datasets["dataset_title"].str.contains("Wilk", case=False, na=False) & datasets[
    "collection_name"
].str.contains("COVID", case=False, na=False)
hit = datasets.loc[mask].iloc[0]
WILK_COLLECTION_ID = hit["collection_id"]
WILK_DATASET_ID = hit["dataset_id"]
print("collection_id", WILK_COLLECTION_ID)
print("dataset_id", WILK_DATASET_ID)
print("title:", hit["dataset_title"])

WILK_FILTER = f"dataset_id == '{WILK_DATASET_ID}'"
print("WILK_FILTER:", WILK_FILTER)


In [ ]:
# Cell 4 — load Wilk + derive severity via explicit donor_id mapping
# Wilk 2020 (Nature Medicine) published donor cohort: H1-H6 healthy, C1-C7 COVID with
# severity flags in the paper's Table S1. CELLxGENE Census does not surface
# disease_severity directly, so we map it here. Halt on any unknown donor_id.
import numpy as np
import pandas as pd

import cellxgene_census as cgc

with cgc.open_soma(census_version=CENSUS_VERSION) as census:
    adata = cgc.get_anndata(
        census,
        organism="Homo sapiens",
        obs_value_filter=WILK_FILTER,
        X_name="raw",
    )

# Explicit Wilk donor -> severity map (Wilk et al. Nat Med 2020 Table S1).
# Adjust this dict if the printed unique donor_ids below do not match.
WILK_DONOR_SEVERITY: dict[str, str] = {
    "H1": "healthy", "H2": "healthy", "H3": "healthy",
    "H4": "healthy", "H5": "healthy", "H6": "healthy",
    "C1": "severe",  "C2": "severe",  "C3": "severe",
    "C4": "severe",  "C5": "severe",  "C6": "severe", "C7": "severe",
    # Non-ICU / ward-level COVID donors in Wilk paper:
    "C1 A": "mild", "C2 A": "mild", "C3 A": "mild", "C4 A": "mild",
    "C5 A": "mild", "C6 A": "mild", "C7 A": "mild",
}

donor = adata.obs["donor_id"].astype(str)
unique_donors = sorted(donor.unique())
print(f"Unique donor_ids in Census ({len(unique_donors)}):", unique_donors)

unknown = set(unique_donors) - set(WILK_DONOR_SEVERITY.keys())
if unknown:
    raise ValueError(
        f"Unmapped donor_ids: {sorted(unknown)}. "
        f"Update WILK_DONOR_SEVERITY with severity from Wilk 2020 Table S1 before proceeding."
    )

adata.obs["disease_severity"] = donor.map(WILK_DONOR_SEVERITY)

# Sanity-check against CELLxGENE's ``disease`` column (COVID-19 vs normal):
# every severity != 'healthy' row must have disease == 'COVID-19'.
cross = pd.crosstab(adata.obs["disease_severity"], adata.obs["disease"])
print("severity x disease crosstab:\n", cross)
bad = adata.obs[(adata.obs["disease_severity"] == "healthy") & (adata.obs["disease"] == "COVID-19")]
if len(bad):
    raise ValueError(f"{len(bad)} cells mapped to 'healthy' but CELLxGENE says disease=COVID-19")

print("Loaded", adata.shape)
print(adata.obs["disease"].value_counts())
xt = pd.crosstab(adata.obs["cell_type"].astype(str).str.slice(0, 40), adata.obs["disease_severity"])
print(xt.iloc[:15, :])


In [ ]:
# Cell 5 — stratified subsample (cell_type × disease_severity), seed 42
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit

y = (
    adata.obs["cell_type"].astype(str)
    + "||"
    + adata.obs["disease_severity"].astype(str)
)
n = min(N_SUBSAMPLE, adata.n_obs)
splitter = StratifiedShuffleSplit(n_splits=1, train_size=n, random_state=SEED)
idx_train, _ = next(splitter.split(np.zeros((adata.n_obs, 1)), y))
adata = adata[idx_train].copy()
print("Subsampled", adata.shape)
print(pd.crosstab(adata.obs["cell_type"].astype(str).str.slice(0, 30), adata.obs["disease_severity"]))


## Step 2: Cell-type taxonomy harmonization


In [ ]:
# Cell 7 — map to PBMC3k 8-class vocabulary + disease_activity
from typing import Literal

import numpy as np
import pandas as pd

PBMC_TYPES = {
    "CD4 T cells",
    "CD14+ Monocytes",
    "B cells",
    "CD8 T cells",
    "NK cells",
    "FCGR3A+ Monocytes",
    "Dendritic cells",
    "Megakaryocytes",
}

CELL_TYPE_MAP: dict[str, str] = {
    "classical monocyte": "CD14+ Monocytes",
    "non-classical monocyte": "FCGR3A+ Monocytes",
    "naive thymus-derived CD4-positive, alpha-beta T cell": "CD4 T cells",
    "central memory CD4-positive, alpha-beta T cell": "CD4 T cells",
    "activated CD4-positive, alpha-beta T cell, human": "CD4 T cells",
    "naive thymus-derived CD8-positive, alpha-beta T cell": "CD8 T cells",
    "effector memory CD8-positive, alpha-beta T cell": "CD8 T cells",
    "memory B cell": "B cells",
    "naive B cell": "B cells",
    "natural killer cell": "NK cells",
    "CD16-negative, CD56-bright natural killer cell, human": "NK cells",
    "conventional dendritic cell": "Dendritic cells",
    "plasmacytoid dendritic cell, human": "Dendritic cells",
    "transitional stage B cell": "B cells",
    "IgA plasmablast": "B cells",
    "IgG plasmablast": "B cells",
    "gamma-delta T cell": "CD4 T cells",
    "neutrophil": "Other",
    "platelet": "Megakaryocytes",
    "erythrocyte": "Other",
}

SEVERITY_MAP: dict[str, Literal["healthy", "mild", "severe"]] = {
    "healthy": "healthy",
    "normal": "healthy",
    "mild": "mild",
    "moderate": "mild",
    "convalescent": "mild",
    "severe": "severe",
    "critical": "severe",
}

raw_ct = adata.obs["cell_type"].astype(str)
adata.obs["cell_type_pbmc"] = raw_ct.map(CELL_TYPE_MAP).fillna("Other")
adata.obs["disease_activity"] = adata.obs["disease_severity"].map(SEVERITY_MAP)
unmapped = adata.obs["disease_activity"].isna().sum()
if unmapped:
    raise ValueError(f"Unmapped disease_severity values: {adata.obs.loc[adata.obs['disease_activity'].isna(), 'disease_severity'].unique()}")

print("CELL_TYPE_MAP sample:", list(CELL_TYPE_MAP.items())[:5])
print("cell_type_pbmc counts:\n", adata.obs["cell_type_pbmc"].value_counts())
print("disease_activity counts:\n", adata.obs["disease_activity"].value_counts())


## Step 3: Geneformer inference


In [ ]:
# Cell 9 — Helical Geneformer + embedding DataFrame
import numpy as np
import pandas as pd
import scanpy as sc
import torch
from helical.models.geneformer import Geneformer, GeneformerConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
model_config = GeneformerConfig(batch_size=8, device=device)
geneformer = Geneformer(configurer=model_config)
print("Geneformer initialized on", device)

# Census stores HGNC symbols in ``feature_name``; ``var_names`` holds Ensembl IDs.
# Geneformer's tokenizer maps HGNC → Ensembl internally, so we must feed it symbols.
# Without this mapping, ``process_data`` matches 0 of ~22k genes and raises.
if "feature_name" in adata.var.columns:
    adata.var["gene_name"] = adata.var["feature_name"].astype(str)
elif "gene_symbols" in adata.var.columns:
    adata.var["gene_name"] = adata.var["gene_symbols"].astype(str)
else:
    # Fallback: assume var_names really are symbols (works for PBMC3k, not for Census).
    adata.var["gene_name"] = adata.var_names.astype(str)

sample_genes = adata.var["gene_name"].head().tolist()
assert not any(g.startswith("ENSG") for g in sample_genes), (
    f"gene_name still contains Ensembl IDs: {sample_genes}. "
    "Check adata.var columns — Census should expose 'feature_name'."
)
print("gene_name sample:", sample_genes)

sc.pp.filter_genes(adata, min_cells=3)

processed = geneformer.process_data(adata, gene_names="gene_name")
embeddings = geneformer.get_embeddings(processed)
emb_arr = np.asarray(embeddings)
if emb_arr.shape[1] != EMBED_DIM:
    raise ValueError(f"Expected dim {EMBED_DIM}, got {emb_arr.shape}")
adata.obsm["X_geneformer"] = emb_arr
print("X_geneformer", adata.obsm["X_geneformer"].shape, "mean", float(emb_arr.mean()), "std", float(emb_arr.std()))

cols = {f"embedding_{i}": emb_arr[:, i].astype(np.float32) for i in range(EMBED_DIM)}
df_covid_embeddings = pd.DataFrame(
    {
        "cell_id": adata.obs_names.astype(str),
        "cell_type": adata.obs["cell_type_pbmc"].values,
        **cols,
    }
)
assert df_covid_embeddings.shape[1] == 514
print(df_covid_embeddings.dtypes.head())


## Step 4: Load PBMC3k reference embeddings + fitted UMAP from S3


In [ ]:
# Cell 11 — reference parquet + UMAP reducer + centroids
import io
import joblib

import boto3
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

if not S3_BUCKET:
    raise ValueError("S3_BUCKET is required")

s3 = boto3.client("s3", region_name=S3_REGION)
key_emb = f"{VERSION}/pbmc3k/geneformer_embeddings.parquet"
key_umap = f"{VERSION}/pbmc3k/pbmc_umap_reducer.joblib"

buf = io.BytesIO()
s3.download_fileobj(S3_BUCKET, key_emb, buf)
buf.seek(0)
table = pq.read_table(buf)
df_pbmc = table.to_pandas()
assert len(df_pbmc) == 2638, len(df_pbmc)

tmp_umap = "/tmp/pbmc_umap_reducer.joblib"
s3.download_file(S3_BUCKET, key_umap, tmp_umap)
pbmc_umap_reducer = joblib.load(tmp_umap)
assert hasattr(pbmc_umap_reducer, "transform"), type(pbmc_umap_reducer)

emb_cols = [c for c in df_pbmc.columns if c.startswith("embedding_")]
centroids = df_pbmc.groupby("cell_type")[emb_cols].mean()
global_centroid = df_pbmc[emb_cols].mean().values.astype(np.float32)
centroid_map = {str(ct): centroids.loc[ct].values.astype(np.float32) for ct in centroids.index}
print("centroids", len(centroid_map), "global", global_centroid.shape)


## Step 5: Project COVID into reference UMAP


In [ ]:
# Cell 13 — transform (no re-fit)
import matplotlib.pyplot as plt
import numpy as np

X = adata.obsm["X_geneformer"]
adata.obsm["X_umap_projected"] = pbmc_umap_reducer.transform(X).astype(np.float32)
assert adata.obsm["X_umap_projected"].shape[1] == 2
print("UMAP projected", adata.obsm["X_umap_projected"].shape)

fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(
    adata.obsm["X_umap_projected"][:, 0],
    adata.obsm["X_umap_projected"][:, 1],
    c=pd.Categorical(adata.obs["cell_type_pbmc"]).codes,
    s=2,
    alpha=0.4,
)
ax.set_title("COVID cells in PBMC3k UMAP basis")
plt.show()


## Step 6: Distance to healthy


In [ ]:
# Cell 15 — cosine distance to nearest centroid + min-max rescale
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_distances

X = adata.obsm["X_geneformer"].astype(np.float32)
dist_raw = np.empty(X.shape[0], dtype=np.float32)
for i, ct in enumerate(adata.obs["cell_type_pbmc"].astype(str)):
    if ct in centroid_map:
        c = centroid_map[ct][None, :]
    else:
        c = global_centroid[None, :]
    dist_raw[i] = cosine_distances(X[i : i + 1], c)[0, 0]

dmin = float(dist_raw.min())
dmax = float(dist_raw.max())
adata.obs["distance_geneformer"] = ((dist_raw - dmin) / (dmax - dmin + 1e-12)).astype(np.float32)
print(adata.obs.groupby("disease_activity")["distance_geneformer"].describe())


## Step 7: Export parquets


In [ ]:
# Cell 17 — PyArrow export + S3 upload (API uses ``distance_scores.parquet`` on disk)
import os
from pathlib import Path

import boto3
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

N = adata.n_obs
umap = adata.obsm["X_umap_projected"]
Xemb = adata.obsm["X_geneformer"].astype(np.float32)
emb_cols = [f"embedding_{i}" for i in range(EMBED_DIM)]

base_rows = {
    "cell_id": adata.obs_names.astype(str),
    "cell_type": adata.obs["cell_type_pbmc"].values,
    "umap_1": umap[:, 0],
    "umap_2": umap[:, 1],
}
emb_dict = {c: Xemb[:, i] for i, c in enumerate(emb_cols)}

schema_emb = pa.schema(
    [("cell_id", pa.string()), ("cell_type", pa.string()), ("umap_1", pa.float32()), ("umap_2", pa.float32())]
    + [(c, pa.float32()) for c in emb_cols]
)
df1 = pd.DataFrame({**base_rows, **emb_dict})
pq.write_table(
    pa.Table.from_pandas(df1, schema=schema_emb, preserve_index=False),
    "/tmp/geneformer_embeddings.parquet",
    compression="snappy",
)

df2 = pd.DataFrame(
    {
        **base_rows,
        "disease_activity": adata.obs["disease_activity"].values,
        "distance_to_healthy": adata.obs["distance_geneformer"].values,
        **emb_dict,
    }
)
schema_proj = pa.schema(
    [
        ("cell_id", pa.string()),
        ("cell_type", pa.string()),
        ("disease_activity", pa.string()),
        ("umap_1", pa.float32()),
        ("umap_2", pa.float32()),
        ("distance_to_healthy", pa.float32()),
    ]
    + [(c, pa.float32()) for c in emb_cols]
)
pq.write_table(
    pa.Table.from_pandas(df2, schema=schema_proj, preserve_index=False),
    "/tmp/geneformer_projected.parquet",
    compression="snappy",
)

dist_genept = np.full(N, np.nan, dtype=np.float32)
df3 = pd.DataFrame(
    {
        "cell_id": adata.obs_names.astype(str),
        "cell_type": adata.obs["cell_type_pbmc"].values,
        "disease_activity": adata.obs["disease_activity"].values,
        "distance_geneformer": adata.obs["distance_geneformer"].values,
        "distance_genept": dist_genept,
    }
)
schema_scores = pa.schema(
    [
        ("cell_id", pa.string()),
        ("cell_type", pa.string()),
        ("disease_activity", pa.string()),
        ("distance_geneformer", pa.float32()),
        ("distance_genept", pa.float32()),
    ]
)
pq.write_table(
    pa.Table.from_pandas(df3, schema=schema_scores, preserve_index=False),
    "/tmp/distance_scores.parquet",
    compression="snappy",
)

for p in [
    "/tmp/geneformer_embeddings.parquet",
    "/tmp/geneformer_projected.parquet",
    "/tmp/distance_scores.parquet",
]:
    print(p, Path(p).stat().st_size // 1024, "KB")

if not S3_BUCKET or not S3_REGION:
    raise ValueError("S3_BUCKET and S3_REGION required for upload")

cli = boto3.client("s3", region_name=S3_REGION)
prefix = f"{VERSION}/covid_wilk"
for name in ["geneformer_embeddings.parquet", "geneformer_projected.parquet", "distance_scores.parquet"]:
    local = f"/tmp/{name}"
    key = f"{prefix}/{name}"
    cli.upload_file(local, S3_BUCKET, key)
    print(f"s3://{S3_BUCKET}/{key}", Path(local).stat().st_size // 1024, "KB")


## Step 8: Provenance


In [ ]:
# Cell 19 — two precompute_runs rows (inline SQLModel, same pattern as precompute_pbmc_mvp)
import asyncio
import ssl
import uuid
from datetime import datetime, timezone
from typing import Optional
from urllib.parse import parse_qs, urlencode, urlparse, urlunparse

from sqlalchemy import Column
from sqlalchemy.dialects.postgresql import JSONB
from sqlalchemy.ext.asyncio import async_sessionmaker, create_async_engine
from sqlmodel import Field, SQLModel, select
from sqlmodel.ext.asyncio.session import AsyncSession as SQLModelAsyncSession


def _prepare_asyncpg_url(raw_url: str) -> tuple:
    connect_args = {"statement_cache_size": 0}
    parsed = urlparse(raw_url)
    params = parse_qs(parsed.query, keep_blank_values=True)
    sslmode = params.pop("sslmode", [None])[0]
    if sslmode and sslmode in ("require", "verify-ca", "verify-full"):
        connect_args["ssl"] = ssl.create_default_context()
    params.pop("channel_binding", None)
    clean_query = urlencode({k: v[0] for k, v in params.items()}, doseq=False)
    clean_url = urlunparse(parsed._replace(query=clean_query))
    return clean_url, connect_args


async def _write_provenance() -> None:
    # ``extend_existing=True`` lets the cell be re-run in the same kernel without
    # tripping SQLAlchemy's "Table already defined" InvalidRequestError.
    class Dataset(SQLModel, table=True):
        __tablename__ = "datasets"
        __table_args__ = {"extend_existing": True}
        id: Optional[uuid.UUID] = Field(default_factory=uuid.uuid4, primary_key=True)
        slug: str = Field(index=True)
        display_name: str = ""
        citation: str = ""
        license: str = ""
        cell_count: int = 0
        gene_count: int = 0

    class PrecomputeRun(SQLModel, table=True):
        __tablename__ = "precompute_runs"
        __table_args__ = {"extend_existing": True}
        id: Optional[uuid.UUID] = Field(default_factory=uuid.uuid4, primary_key=True)
        dataset_id: uuid.UUID = Field(foreign_key="datasets.id", index=True)
        model_name: str
        model_version: str
        # SQLModel cannot auto-map a plain ``dict`` — bind to JSONB explicitly.
        parameters: dict = Field(default_factory=dict, sa_column=Column(JSONB, nullable=False))
        output_parquet_key: str
        git_sha: str
        # The ``precompute_runs.created_at`` column is TIMESTAMP WITHOUT TIME ZONE,
        # so strip tzinfo to avoid asyncpg's offset-naive/aware DataError.
        created_at: datetime = Field(
            default_factory=lambda: datetime.now(timezone.utc).replace(tzinfo=None)
        )

    database_url, connect_args = _prepare_asyncpg_url(DATABASE_URL)
    engine = create_async_engine(database_url, echo=False, connect_args=connect_args)
    session_factory = async_sessionmaker(
        engine, class_=SQLModelAsyncSession, expire_on_commit=False
    )
    try:
        async with session_factory() as session:
            result = await session.exec(select(Dataset).where(Dataset.slug == "covid_wilk"))
            dataset = result.first()
            if dataset is None or dataset.id is None:
                raise RuntimeError("No datasets row with slug=covid_wilk — run seed_covid_dataset first.")

            r1 = PrecomputeRun(
                dataset_id=dataset.id,
                model_name="geneformer",
                model_version="v1",
                parameters={
                    "n_subsample": int(N_SUBSAMPLE),
                    "seed": SEED,
                    "source": "cellxgene_census:wilk_2020",
                },
                output_parquet_key=f"{VERSION}/covid_wilk/geneformer_embeddings.parquet",
                git_sha=GIT_SHA[:40],
            )
            r2 = PrecomputeRun(
                dataset_id=dataset.id,
                model_name="geneformer_projection",
                model_version="v1",
                parameters={"reference": "pbmc3k", "metric": "cosine", "k_nearest": 1},
                output_parquet_key=f"{VERSION}/covid_wilk/geneformer_projected.parquet",
                git_sha=GIT_SHA[:40],
            )
            session.add(r1)
            session.add(r2)
            await session.commit()
            await session.refresh(r1)
            await session.refresh(r2)
            print(f"Inserted precompute_runs ids={r1.id}, {r2.id}")
    finally:
        await engine.dispose()


try:
    import nest_asyncio

    nest_asyncio.apply()
    asyncio.get_event_loop().run_until_complete(_write_provenance())
except ImportError:
    await _write_provenance()  # type: ignore[top-level-await,misc]


In [ ]:
# Cell 20 — re-read from S3 + schema checks
import io

import boto3
import numpy as np
import pyarrow.parquet as pq

cli = boto3.client("s3", region_name=S3_REGION)
prefix = f"{VERSION}/covid_wilk"

for name in ["geneformer_embeddings.parquet", "geneformer_projected.parquet", "distance_scores.parquet"]:
    buf = io.BytesIO()
    cli.download_fileobj(S3_BUCKET, f"{prefix}/{name}", buf)
    buf.seek(0)
    t = pq.read_table(buf)
    print(name, t.num_rows, t.schema)
    if name == "distance_scores.parquet":
        col = t.column("distance_genept").to_numpy()
        assert np.isnan(col).all()
    if name != "distance_scores.parquet":
        assert t.num_rows == adata.n_obs


In [ ]:
# Cell 21 — API smoke test
import os

import httpx

base = os.environ.get("BACKEND_URL", "https://api.helical.manumustudio.com").rstrip("/")
urls = [
    f"{base}/api/v1/projections/covid_wilk/geneformer",
    f"{base}/api/v1/scores/covid_wilk",
]
for u in urls:
    r = httpx.get(u, timeout=60.0)
    r.raise_for_status()
    js = r.json()
    print(u, "keys", js.keys(), "sampled", js.get("sampled"), "total", js.get("total_cells"))


## Local fallback copy (after Colab)

```bash
mkdir -p backend/data/parquet/covid_wilk/
cp /tmp/{geneformer_embeddings,geneformer_projected,distance_scores}.parquet backend/data/parquet/covid_wilk/
```
